In [0]:
%run /Users/manoranjanpandey836@gmail.com/Fake_API/config_utility

In [0]:
gold_df = spark.read.table("workspace.silver.finance_transactions_silver")

In [0]:
gold_df.show(10)

### Column level monthly aggregation

In [0]:
import pyspark.sql.functions as F
gold_df = (
    gold_df \
    .groupBy(
        "customer_id",
        "transaction_month"
    )
    .agg(
        #Transction metics
        F.count("transaction_id").alias("total_transactions"),
        F.sum("amount").alias("total_transaction_amount"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.max("amount").alias("max_transaction_amount"),

        #Fraud metrics
        F.sum(
            F.when(F.col("is_fraud") == 1, 1).otherwise(0)
        ).alias("fraud_transaction_count"),

        F.round(
            F.avg(
            F.when(F.col("is_fraud") == 1, 1).otherwise(0)
            ) * 100,
            2
        ).alias("fraud_rate_percent"),

        F.avg("credit_score").alias("avg_credit_score")
    )
)

In [0]:
gold_df.show(10)

In [0]:
def write_to_gold():
    gold_df.write \
        .format("delta") \
        .mode(gold["write_mode"]) \
        .saveAsTable(gold["full_table_name"])

    return True


df_wirte = write_to_gold()

print("gold table created successfully!")
print("Table name:", gold["full_table_name"])

In [0]:
%sql
select * from workspace.gold.finance_transactions_gold order by customer_id desc;